# Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
from itertools import permutations
import requests
from tqdm import tqdm
import os
import pickle
import glob

# Explore all datasets

## METR-LA

In [ ]:
df = pd.read_hdf('data/METRLA/metr-la.h5')
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
total_entries = df.size
zero_entries = (df == 0).sum().sum()
percentage_zeros = (zero_entries / total_entries) * 100
percentage_zeros

## PEMS-BAY

In [ ]:
df2 = pd.read_hdf('data/PEMSBAY/PEMS-BAY.h5')
df2.head()

In [ ]:
df2.tail()

In [ ]:
df2.describe()

In [ ]:
total_entries = df2.size
zero_entries = (df2 == 0).sum().sum()
percentage_zeros = (zero_entries / total_entries) * 100
percentage_zeros

## HAUGE

In [ ]:
df3 = pd.read_hdf('data/Hauge/hague_comp_filled.h5', sep=';')

In [ ]:
df3.head()

In [ ]:
df3.tail()

In [ ]:
df3.describe()

In [ ]:
total_entries = df3.size
zero_entries = (df3 == 0).sum().sum()
percentage_zeros = (zero_entries / total_entries) * 100
percentage_zeros

In [ ]:
df4 = pd.read_hdf('data/Hauge/hague_filled.h5', sep=';')

In [ ]:
df4.describe()

# Calculate new distance

### Illustration of two points on map

In [ ]:
import requests
from geopy.distance import geodesic
import folium
from IPython.display import display

def compute_and_visualize_distances(sensor_a, sensor_b):
    osrm_server = "http://router.project-osrm.org"

    # Compute the direct distance using the Haversine formula
    direct_distance = geodesic((sensor_a['latitude'], sensor_a['longitude']), (sensor_b['latitude'], sensor_b['longitude'])).meters
    
    # Compute the routing distance using the OSRM API
    request_url = f"{osrm_server}/route/v1/driving/{sensor_a['longitude']},{sensor_a['latitude']};{sensor_b['longitude']},{sensor_b['latitude']}?overview=full&geometries=geojson"
    response = requests.get(request_url)
    route_data = response.json()
    
    if route_data['code'] == 'Ok':
        routing_distance = route_data['routes'][0]['distance']  # Distance in meters
        route_geometry = route_data['routes'][0]['geometry']
    else:
        routing_distance = float('inf')  # An arbitrary large number to denote failure to get distance
        route_geometry = None
    
    # Create a map centered around the midpoint of the two sensors
    midpoint = ((sensor_a['latitude'] + sensor_b['latitude']) / 2, (sensor_a['longitude'] + sensor_b['longitude']) / 2)
    m = folium.Map(location=midpoint, zoom_start=13, tiles='CartoDB Positron')
    
    # Add markers for the two sensors
    folium.Marker([sensor_a['latitude'], sensor_a['longitude']], popup=f"Sensor A: {sensor_a['sensor_id']}").add_to(m)
    folium.Marker([sensor_b['latitude'], sensor_b['longitude']], popup=f"Sensor B: {sensor_b['sensor_id']}").add_to(m)
    
    # Draw a line for the direct distance
    folium.PolyLine(
        locations=[(sensor_a['latitude'], sensor_a['longitude']), (sensor_b['latitude'], sensor_b['longitude'])],
        color='blue',
        weight=2,
        opacity=0.6,
        tooltip=f"Direct distance: {direct_distance:.2f} meters"
    ).add_to(m)
    
    # Draw a line for the routing distance if available
    if route_geometry:
        folium.PolyLine(
            locations=[(coord[1], coord[0]) for coord in route_geometry['coordinates']],
            color='green',
            weight=2,
            opacity=0.6,
            tooltip=f"Routing distance: {routing_distance:.2f} meters"
        ).add_to(m)
    
    # Display the map
    display(m)

    # Print distances
    print(f"Direct distance: {direct_distance:.2f} meters")
    print(f"Routing distance: {routing_distance:.2f} meters")


In [ ]:
sensor_a = {'sensor_id': 9, 'latitude': 34.15562, 'longitude': -118.46860}  # Example coordinates for sensor A
sensor_b = {'sensor_id': 75, 'latitude': 34.15571, 'longitude': -118.43273}  # Example coordinates for sensor B

compute_and_visualize_distances(sensor_a, sensor_b)

## Hague

In [ ]:
def read_sensors_and_group_by_trajectory(csv_file_path):
    trajectory_groups = {'T1N': [], 'T1S': [], 'T2N': [], 'T2S': []}
    with open(csv_file_path, mode='r') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            trajectories = row['trajectory'].split('+')
            for trajectory in trajectories:
                trajectory_groups[trajectory].append(row)
    return trajectory_groups

def calculate_distances_osrm_grouped(sensors_grouped):
    # osrm_server = "http://router.project-osrm.org"
    osrm_server = "http://127.0.0.1:5001"
    results = []

    for trajectory, sensors in sensors_grouped.items():
        # Prepare the combinations and wrap it with tqdm for the progress bar
        sensor_pairs = list(permutations(sensors, 2))
        for sensor_a, sensor_b in tqdm(sensor_pairs, desc=f"Calculating distances for {trajectory}"):
            request_url = f"{osrm_server}/route/v1/driving/{sensor_a['longitude']},{sensor_a['latitude']};{sensor_b['longitude']},{sensor_b['latitude']}?overview=false"
            
            response = requests.get(request_url)
            route_data = response.json()
            
            if route_data['code'] == 'Ok':
                cost = route_data['routes'][0]['distance']  # Distance in meters
            else:
                cost = float('inf')  # Indicate failure
            
            results.append({
                'trajectory': trajectory,
                'from': sensor_a['sensor_id'],
                'to': sensor_b['sensor_id'],
                'cost': cost
            })
            # print(f"From {sensor_a['sensor_id']} to {sensor_b['sensor_id']} cost: {cost}")

    return results


In [ ]:
csv_file_path = 'data/Hauge/location_std.csv'
sensors_grouped = read_sensors_and_group_by_trajectory(csv_file_path)
results = calculate_distances_osrm_grouped(sensors_grouped)


Save distances betwee each sensor on the same trajectory into a csv file

In [ ]:
with open('data/Hauge/distances.csv', mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=['trajectory', 'from', 'to', 'cost'])
    writer.writeheader()
    writer.writerows(results)

## METR-LA

In [ ]:
def read_sensors(csv_file_path):
    sensors = []
    with open(csv_file_path, mode='r') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            sensors.append(row)
    return sensors

In [ ]:
import requests
from tqdm import tqdm

def calculate_distances_osrm(sensors):
    results = []
    osrm_server = "http://127.0.0.1:5000"
    
    for sensor_a in tqdm(sensors, desc="Calculating distances"):
        for sensor_b in sensors:
            if sensor_a['sensor_id'] == sensor_b['sensor_id']:
                cost = 0.0  # Cost is zero when it's the same sensor
            else:
                # Constructing the request URL
                request_url = f"{osrm_server}/route/v1/driving/{sensor_a['longitude']},{sensor_a['latitude']};{sensor_b['longitude']},{sensor_b['latitude']}?overview=false"
                print(request_url)
                
                try:
                    # Making the request to the OSRM API
                    response = requests.get(request_url)
                    route_data = response.json()
                    
                    # Parsing the distance from the response
                    if route_data['code'] == 'Ok':
                        cost = route_data['routes'][0]['distance']  # Distance in meters
                    else:
                        cost = float('inf')  # An arbitrary large number to denote failure to get distance
                except requests.exceptions.RequestException as e:
                    print(f"Error with request between sensor {sensor_a['sensor_id']} and sensor {sensor_b['sensor_id']}: {e}")
                    cost = float('inf')  # Set the cost to infinity on error
                
            results.append({'from': sensor_a['sensor_id'], 'to': sensor_b['sensor_id'], 'cost': cost})
    
    return results


In [ ]:
csv_file_path = 'data/METRLA/graph_sensor_locations.csv'
output_csv_file_path = 'data/METRLA/distances_la_route.csv'

sensors = read_sensors(csv_file_path)
distance_results = calculate_distances_osrm(sensors)

In [ ]:
with open(output_csv_file_path, mode='w', newline='') as csvfile:
    fieldnames = ['from', 'to', 'cost']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for result in distance_results:
        writer.writerow(result)

print(f"Distances saved to {output_csv_file_path}")

## PeMS-BAY

In [ ]:
csv_file_path = 'data/PEMSBAY/graph_sensor_locations_bay.csv'
output_csv_file_path = 'data/PEMSBAY/distances_bay_route.csv'
sensors = read_sensors(csv_file_path)
distance_results = calculate_distances_osrm(sensors)

In [ ]:
with open(output_csv_file_path, mode='w', newline='') as csvfile:
    fieldnames = ['from', 'to', 'cost']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for result in distance_results:
        writer.writerow(result)

print(f"Distances saved to {output_csv_file_path}")

## PeMSD7

In [ ]:
csv_file_path = 'data/PEMSD7/PeMSD7_M_Station_Info.csv'
output_csv_file_path = 'data/PEMSD7/graph_sensor_locations_d7.csv'

# Read the CSV file
df = pd.read_csv(csv_file_path)

# Select and rename the required columns
df_filtered = df[['ID', 'Latitude', 'Longitude']].rename(columns={'ID': 'sensor_id', 'Latitude': 'latitude', 'Longitude': 'longitude'})

# Save the new DataFrame to a CSV file
df_filtered.to_csv(output_csv_file_path, index=False)

print(f"Filtered data saved to {output_csv_file_path}")

In [ ]:
csv_file_path = 'data/PEMSD7/graph_sensor_locations_d7.csv'
output_csv_file_path = 'data/PEMSD7/distances_d7_route.csv'
sensors = read_sensors(csv_file_path)
distance_results = calculate_distances_osrm(sensors)

In [ ]:
with open(output_csv_file_path, mode='w', newline='') as csvfile:
    fieldnames = ['from', 'to', 'cost']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for result in distance_results:
        writer.writerow(result)

print(f"Distances saved to {output_csv_file_path}")

In [ ]:
import csv
import requests

def retry_inf_distances_in_csv(input_csv_file_path, output_csv_file_path, sensors, osrm_server="http://127.0.0.1:5000"):
    # Read the existing CSV file and store the data
    results = []
    with open(input_csv_file_path, mode='r', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            # Convert cost to float, handling 'inf' as a string
            row['cost'] = float(row['cost']) if row['cost'] != 'inf' else float('inf')
            results.append(row)
    
    # Retry the failed (inf) cost entries
    for result in results:
        if result['cost'] == float('inf'):
            sensor_a = next(s for s in sensors if s['sensor_id'] == result['from'])
            sensor_b = next(s for s in sensors if s['sensor_id'] == result['to'])
            
            # Retry the request for this pair
            request_url = f"{osrm_server}/route/v1/driving/{sensor_a['longitude']},{sensor_a['latitude']};{sensor_b['longitude']},{sensor_b['latitude']}?overview=false"
            try:
                response = requests.get(request_url)
                route_data = response.json()
                
                if route_data['code'] == 'Ok':
                    result['cost'] = route_data['routes'][0]['distance']  # Update the cost
                else:
                    result['cost'] = float('inf')  # If still 'inf', leave it as it is
            except requests.exceptions.RequestException as e:
                print(f"Retry failed for {sensor_a['sensor_id']} to {sensor_b['sensor_id']}: {e}")
                result['cost'] = float('inf')
    
    # Write the updated results back to the CSV file
    with open(output_csv_file_path, mode='w', newline='') as csvfile:
        fieldnames = ['from', 'to', 'cost']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        writer.writeheader()
        for result in results:
            writer.writerow(result)

    print(f"Updated distances saved to {output_csv_file_path}")

retry_inf_distances_in_csv(output_csv_file_path,'data/PEMSD7/updated_distances_d7_route.csv', sensors)

# Create new adjacency matrix

## METR-LA

In [ ]:
distances_df = pd.read_csv('data/METRLA/distances_la_route.csv')
df = pd.read_hdf('data/METRLA/metr-la.h5')
df.head()

In [ ]:
def get_adjacency_matrix(dist_mx_array, normalized_k=0.1):
    # Calculate the standard deviation as theta for normalization
    distances = dist_mx_array[~np.isinf(dist_mx_array)].flatten()
    std = distances.std()
    adj_mx_array = np.exp(-np.square(dist_mx_array / std))
    
    # Set entries below a threshold to zero for sparsity
    adj_mx_array[adj_mx_array < normalized_k] = 0
    return adj_mx_array

In [ ]:
sensor_ids = df.columns.astype(str).tolist()

# Efficiently convert sensor IDs to index for quick lookup (now all as strings)
sensor_id_to_index = {str(sensor_id): index for index, sensor_id in enumerate(sensor_ids)}

# Initialize an empty distance matrix
num_sensors = len(sensor_ids)
dist_mx_array = np.full((num_sensors, num_sensors), np.inf)

# Make sure 'from' and 'to' in distances_df are also strings
distances_df['from'] = distances_df['from'].astype(str)
distances_df['to'] = distances_df['to'].astype(str)


In [ ]:
# Populate the distance matrix with the data from distances_df
for _, row in distances_df.iterrows():
    from_sensor = row['from']
    to_sensor = row['to']
    if from_sensor in sensor_id_to_index and to_sensor in sensor_id_to_index:
        i = sensor_id_to_index[from_sensor]
        j = sensor_id_to_index[to_sensor]
        dist_mx_array[i, j] = row['cost']

# Replace the diagonal with zeros since the distance from a sensor to itself is zero
np.fill_diagonal(dist_mx_array, 0)

In [ ]:
adj_mx_array = get_adjacency_matrix(dist_mx_array).astype(np.float32)
adj_mx = [sensor_ids, sensor_id_to_index, adj_mx_array]

In [ ]:
directory = 'data/METRLA'
file_name = 'adj_mx_new.pkl'
file_path = os.path.join(directory, file_name)

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# Save the adj_mx to a .pkl file
with open(file_path, 'wb') as f:
    pickle.dump(adj_mx, f)

## PeMS-BAY

In [ ]:
# Load distances from CSV
distances_df = pd.read_csv('data/PEMSBAY/distances_bay_route.csv')
df = pd.read_hdf('data/PEMSBAY/pems-bay.h5')

In [ ]:
def get_adjacency_matrix(dist_mx_array, normalized_k=0.1):
    # Calculate the standard deviation as theta for normalization
    distances = dist_mx_array[~np.isinf(dist_mx_array)].flatten()
    std = distances.std()
    adj_mx_array = np.exp(-np.square(dist_mx_array / std))
    
    # Set entries below a threshold to zero for sparsity
    adj_mx_array[adj_mx_array < normalized_k] = 0
    return adj_mx_array

In [ ]:
sensor_ids = df.columns.astype(str).tolist()

# Efficiently convert sensor IDs to index for quick lookup (now all as strings)
sensor_id_to_index = {str(sensor_id): index for index, sensor_id in enumerate(sensor_ids)}

# Initialize an empty distance matrix
num_sensors = len(sensor_ids)
dist_mx_array = np.full((num_sensors, num_sensors), np.inf)

# Make sure 'from' and 'to' in distances_df are also strings
distances_df['from'] = distances_df['from'].astype(str)
distances_df['to'] = distances_df['to'].astype(str)


In [ ]:
# Populate the distance matrix with the data from distances_df
for _, row in distances_df.iterrows():
    from_sensor = row['from']
    to_sensor = row['to']
    if from_sensor in sensor_id_to_index and to_sensor in sensor_id_to_index:
        i = sensor_id_to_index[from_sensor]
        j = sensor_id_to_index[to_sensor]
        dist_mx_array[i, j] = row['cost']

# Replace the diagonal with zeros since the distance from a sensor to itself is zero
np.fill_diagonal(dist_mx_array, 0)

In [ ]:
adj_mx_array = get_adjacency_matrix(dist_mx_array).astype(np.float32)

# Construct the adj_mx list as specified with the updated adj_mx_array
adj_mx = [sensor_ids, sensor_id_to_index, adj_mx_array]

In [ ]:
directory = 'data/PEMSBAY'
file_name = 'adj_mx_new.pkl'
file_path = os.path.join(directory, file_name)

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# Save the adj_mx to a .pkl file
with open(file_path, 'wb') as f:
    pickle.dump(adj_mx, f)

# Creating h5 file for PeMS-D7 

In [76]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import BDay

# Step 1: Load the CSV files
sensor_data = pd.read_csv('data/PEMSD7/PeMSD7_Full/PeMSD7_V_228.csv', header=None)
sensor_info = pd.read_csv('data/PEMSD7/PeMSD7_M_Station_Info.csv')

# Step 2: Create the timestamp range (Weekdays between May to June 2012 with 5-minute intervals)
dates = pd.date_range('2012-05-01', '2012-06-30', freq='B')  # Weekdays only (B)
timestamps = pd.date_range(start='2012-05-01 00:00:00', periods=len(dates) * 288, freq='5T')

# Step 3: Assign column names from the sensor_info file
sensor_ids = sensor_info['ID'].astype(str).values
sensor_data.columns = sensor_ids

# Step 4: Make sure the number of rows matches the timestamps
sensor_data.index = timestamps

# Step 5: Save data to an HDF5 file using Pandas' HDFStore for compatibility
with pd.HDFStore('data/PEMSD7/pemsd7.h5', 'w') as store:
    # Save the sensor data (table format)
    store.put('sensor_readings', sensor_data)

print("HDF5 file created successfully!")

# Read and display the contents of the newly created HDF5 file
df = pd.read_hdf('data/PEMSD7/pemsd7.h5', 'sensor_readings')
df.head()


HDF5 file created successfully!


C:\Users\erays\AppData\Local\Temp\ipykernel_23324\2327851776.py:11: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  timestamps = pd.date_range(start='2012-05-01 00:00:00', periods=len(dates) * 288, freq='5T')


,716939,759700,716925,715930,715938,716908,716912,759707,716930,764232,...,716484,763400,763413,718164,763392,766337,763272,763532,763522,763275
2012-05-01 00:00:00,71.1,66.0,64.6,65.6,67.1,71.9,68.6,67.7,65.8,40.9,...,69.1,70.9,65.0,64.5,66.6,66.6,65.0,69.3,67.7,68.9
2012-05-01 00:05:00,68.1,66.8,61.7,66.7,64.5,71.6,72.3,64.9,65.6,40.1,...,70.6,65.4,65.0,64.9,65.1,67.7,65.0,67.7,68.8,68.8
2012-05-01 00:10:00,68.0,64.3,66.6,68.7,68.1,70.5,70.2,61.7,63.4,39.6,...,72.2,70.5,65.0,64.7,66.7,68.9,65.0,70.2,69.1,68.7
2012-05-01 00:15:00,68.3,67.8,65.9,66.6,67.9,70.3,69.8,67.6,63.2,37.6,...,71.2,69.7,65.0,65.2,67.2,66.9,65.0,70.4,67.3,69.0
2012-05-01 00:20:00,68.9,69.5,61.2,67.4,64.0,68.1,67.0,66.7,64.2,36.8,...,71.3,65.8,65.0,66.3,66.7,66.2,65.0,68.0,67.4,68.1


In [64]:
df.tail()

,716939,759700,716925,715930,715938,716908,716912,759707,716930,764232,...,716484,763400,763413,718164,763392,766337,763272,763532,763522,763275
2012-06-13 23:35:00,70.3,65.9,70.2,62.1,66.8,66.0,64.0,64.8,24.6,66.5,...,66.0,55.6,29.9,63.8,64.5,62.8,3.8,70.2,68.1,19.3
2012-06-13 23:40:00,69.9,54.0,68.2,62.2,67.4,65.8,64.4,61.3,35.1,69.2,...,66.8,62.8,29.9,62.9,66.6,63.0,3.5,68.2,68.3,18.9
2012-06-13 23:45:00,68.9,37.9,68.8,66.3,69.4,66.6,65.0,60.1,38.5,68.7,...,66.4,65.4,29.9,66.3,68.1,62.6,3.5,68.7,67.5,19.7
2012-06-13 23:50:00,69.2,37.8,68.7,63.6,68.5,66.2,64.1,60.9,40.5,68.5,...,67.6,68.9,30.2,64.8,68.2,63.1,3.7,67.7,67.4,19.5
2012-06-13 23:55:00,68.6,52.9,68.6,65.5,69.3,66.1,64.1,63.5,43.6,58.6,...,66.6,68.3,30.5,64.7,69.1,61.8,4.0,68.1,68.5,19.1


## PeMS-D7

In [1]:
# Load distances from CSV
distances_df = pd.read_csv('data/PEMSD7/updated_distances_d7_route.csv')
df = pd.read_hdf('data/PEMSD7/pemsd7.h5')
df.head()

NameError: name 'pd' is not defined

In [ ]:
def get_adjacency_matrix(dist_mx_array, normalized_k=0.1):
    # Calculate the standard deviation as theta for normalization
    distances = dist_mx_array[~np.isinf(dist_mx_array)].flatten()
    std = distances.std()
    adj_mx_array = np.exp(-np.square(dist_mx_array / std))
    
    # Set entries below a threshold to zero for sparsity
    adj_mx_array[adj_mx_array < normalized_k] = 0
    return adj_mx_array

In [ ]:
sensor_ids = df.columns.astype(str).tolist()

# Efficiently convert sensor IDs to index for quick lookup (now all as strings)
sensor_id_to_index = {str(sensor_id): index for index, sensor_id in enumerate(sensor_ids)}

# Initialize an empty distance matrix
num_sensors = len(sensor_ids)
dist_mx_array = np.full((num_sensors, num_sensors), np.inf)

# Make sure 'from' and 'to' in distances_df are also strings
distances_df['from'] = distances_df['from'].astype(str)
distances_df['to'] = distances_df['to'].astype(str)

In [ ]:
# Populate the distance matrix with the data from distances_df
for _, row in distances_df.iterrows():
    from_sensor = row['from']
    to_sensor = row['to']
    if from_sensor in sensor_id_to_index and to_sensor in sensor_id_to_index:
        i = sensor_id_to_index[from_sensor]
        j = sensor_id_to_index[to_sensor]
        dist_mx_array[i, j] = row['cost']

# Replace the diagonal with zeros since the distance from a sensor to itself is zero
np.fill_diagonal(dist_mx_array, 0)

In [ ]:
adj_mx_array = get_adjacency_matrix(dist_mx_array).astype(np.float32)

# Construct the adj_mx list as specified with the updated adj_mx_array
adj_mx = [sensor_ids, sensor_id_to_index, adj_mx_array]

In [ ]:
directory = 'data/PEMSD7'
file_name = 'adj_mx_new.pkl'
file_path = os.path.join(directory, file_name)

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# Save the adj_mx to a .pkl file
with open(file_path, 'wb') as f:
    pickle.dump(adj_mx, f)

In [ ]:
file_path = 'data/PEMSD7/adj_mx_new.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/PEMSD7/adj_mx_new.csv", adj_mx[2], delimiter=",", fmt='%.4f')

# SIMULATION GRID

In [9]:
import re
import pandas as pd
from datetime import timedelta

# =================== EDIT THESE PATHS/SETTINGS ===================
INPUT_CSV = r"data\SIM_GRID\CLOSURE\closure_merged_all_days_by_sensor.csv"  # your single wide CSV
OUT_H5    = r"data\SIM_GRID\CLOSURE\sensor_readings.h5"
HDF_KEY   = "sensor_readings"

# Start date for building timestamps (day_01 -> this date, day_02 -> +1 day, etc.)
START_DATE = pd.Timestamp("2012-05-01 00:00:00")

# Expected time step encoded in headers like day_01_0, day_01_5, ...
STEP_MIN = 5
# ================================================================

def parse_time_cols(cols):
    """
    From column names like 'day_01_0', return a DataFrame with:
      col_name, day (int), minute (int), timestamp (pd.Timestamp)
    """
    time_meta = []
    rx = re.compile(r"^day_(\d+)_(\d+)$")
    for c in cols:
        m = rx.match(c)
        if not m:
            continue
        day = int(m.group(1))
        minute = int(m.group(2))
        ts = START_DATE + timedelta(days=day - 1, minutes=minute)
        time_meta.append((c, day, minute, ts))
    if not time_meta:
        raise ValueError("No time columns matched pattern 'day_XX_MIN'. Check your header names.")
    meta_df = pd.DataFrame(time_meta, columns=["col", "day", "minute", "timestamp"])
    meta_df = meta_df.sort_values(["day", "minute"]).reset_index(drop=True)
    return meta_df

# ---- Load and transform ----
# Auto-detect comma/tab delimiter
df_wide = pd.read_csv(INPUT_CSV, sep=None, engine="python")

if "sensor_id" not in df_wide.columns:
    raise ValueError("Expected a 'sensor_id' column.")

# Ensure sensor_id is string and unique (aggregate duplicates just in case)
df_wide["sensor_id"] = df_wide["sensor_id"].astype(str)
if df_wide["sensor_id"].duplicated().any():
    df_wide = df_wide.groupby("sensor_id", as_index=False).sum(numeric_only=True)

# Identify time columns and build timestamps
time_meta = parse_time_cols([c for c in df_wide.columns if c != "sensor_id"])
ordered_time_cols = time_meta["col"].tolist()

# Keep only sensor_id + ordered time columns; set sensor_id as index
df_wide = df_wide[["sensor_id"] + ordered_time_cols].set_index("sensor_id")

# Force numeric readings (coerce any bad cells to NaN)
df_wide[ordered_time_cols] = df_wide[ordered_time_cols].apply(pd.to_numeric, errors="coerce")

# Pivot to PeMS layout: rows=timestamps, cols=sensors
data = df_wide[ordered_time_cols].T
data.index = pd.DatetimeIndex(time_meta["timestamp"].values, name="timestamp")
data = data.reindex(sorted(data.columns), axis=1)  # optional stable column order

# ---- Save to HDF5 ----
with pd.HDFStore(OUT_H5, mode="w") as store:
    store.put(HDF_KEY, data, format="table", data_columns=["timestamp"], complevel=9, complib="zlib")

print(f"✅ HDF5 created at: {OUT_H5}")
print(f"   Key: /{HDF_KEY}, shape: {data.shape}")
print(f"   Time range: {data.index.min()} → {data.index.max()}")

# Optional peek
display(data.head())


✅ HDF5 created at: data\SIM_GRID\CLOSURE\sensor_readings.h5
   Key: /sensor_readings, shape: (8670, 120)
   Time range: 2012-05-01 00:00:00 → 2012-05-31 00:00:00


sensor_id,det_A0A1_0,det_A0B0_0,det_A1A0_0,det_A1A2_0,det_A1B1_0,det_A2A1_0,det_A2A3_0,det_A2B2_0,det_A3A2_0,det_A3A4_0,...,det_F2F1_0,det_F2F3_0,det_F3E3_0,det_F3F2_0,det_F3F4_0,det_F4E4_0,det_F4F3_0,det_F4F5_0,det_F5E5_0,det_F5F4_0
timestamp,,,,,,,,,,,,,,,,,,,,,
2012-05-01 00:00:00,1,3,3,6,4,2,5,8,4,4,...,0,1,4,1,0,3,1,0,0,1
2012-05-01 00:05:00,2,0,0,3,3,0,4,7,5,2,...,0,0,5,2,0,7,0,1,1,2
2012-05-01 00:10:00,2,3,3,6,8,4,3,5,7,1,...,1,4,4,1,0,7,2,0,0,1
2012-05-01 00:15:00,1,1,1,5,5,1,3,7,6,0,...,0,1,2,1,0,6,2,0,0,0
2012-05-01 00:20:00,2,3,3,8,7,8,6,8,5,4,...,0,4,5,1,0,11,3,0,0,1


In [12]:
import os
import re
import pickle
import numpy as np
import pandas as pd

# ========== EDIT THESE ==========
DIST_CSV = r"data\SIM_GRID\CLOSURE\sensor_distance_matrix.csv"  # your distance matrix
OUT_DIR  = r"data\SIM_GRID\CLOSURE"                                           # where to save pickles
BASENAME = "adj_mx"                                                 # file stem
THRESHOLD = 0.1                                                     # normalized_k
# ================================

os.makedirs(OUT_DIR, exist_ok=True)

def get_adjacency_matrix(dist_mx_array, normalized_k=0.1):
    distances = dist_mx_array[~np.isinf(dist_mx_array)].flatten()
    std = distances.std()
    if std == 0:
        adj = np.exp(-(dist_mx_array / 1.0)**2)
    else:
        adj = np.exp(-np.square(dist_mx_array / std))
    adj[adj < normalized_k] = 0
    return adj

# ---- Load distance CSV and detect sensor IDs ----
df = pd.read_csv(DIST_CSV)

first_col = df.columns[0]
looks_like_id_col = (
    df[first_col].dtype == object
    or df[first_col].astype(str).str.contains(r"[A-Za-z_]").any()
)

if looks_like_id_col:
    sensor_ids = df[first_col].astype(str).tolist()
    df2 = df.set_index(first_col)
    # if columns are also sensor ids but shuffled, reorder to row order
    if set(df2.columns) == set(sensor_ids):
        df2 = df2.reindex(index=sensor_ids, columns=sensor_ids)
    dist = df2.to_numpy(dtype=float)
else:
    # no explicit id col -> use header columns if stringy, else 0..N-1
    col_labels = list(df.columns)
    if any(not re.match(r"^\d+$", str(c)) for c in col_labels):
        sensor_ids = [str(c) for c in col_labels]
    else:
        sensor_ids = [str(i) for i in range(len(df))]
    dist = df.to_numpy(dtype=float)

# sanitize distance matrix
dist = np.array(dist, dtype=float)
np.fill_diagonal(dist, 0.0)
dist[~np.isfinite(dist)] = np.inf
dist[dist < 0] = np.inf

# ---- Build adjacency (thresholded RBF using your spec) ----
adj_base = get_adjacency_matrix(dist, normalized_k=THRESHOLD).astype(np.float32)

# version A: diagonal = 0
adj_diag0 = adj_base.copy()
np.fill_diagonal(adj_diag0, 0.0)

# version B: diagonal = 1 (self-loops)
adj_diag1 = adj_base.copy()
np.fill_diagonal(adj_diag1, 1.0)

# ---- Pack in the exact list format you showed ----
sensor_id_to_index = {sid: i for i, sid in enumerate(sensor_ids)}
adj_list_diag0 = [sensor_ids, sensor_id_to_index, adj_diag0]
adj_list_diag1 = [sensor_ids, sensor_id_to_index, adj_diag1]

# ---- Save pickles ----
pkl0 = os.path.join(OUT_DIR, f"{BASENAME}_diag0.pkl")
pkl1 = os.path.join(OUT_DIR, f"{BASENAME}_diag1.pkl")

with open(pkl0, "wb") as f:
    pickle.dump(adj_list_diag0, f, protocol=4)
with open(pkl1, "wb") as f:
    pickle.dump(adj_list_diag1, f, protocol=4)

# (optional) CSVs for inspection
pd.DataFrame(adj_diag0, index=sensor_ids, columns=sensor_ids).to_csv(
    os.path.join(OUT_DIR, f"{BASENAME}_diag0.csv")
)
pd.DataFrame(adj_diag1, index=sensor_ids, columns=sensor_ids).to_csv(
    os.path.join(OUT_DIR, f"{BASENAME}_diag1.csv")
)

print("✅ wrote:")
print(" ", pkl0)
print(" ", pkl1)
print(f"sensors: {len(sensor_ids)}, adj shape: {adj_diag0.shape}")


✅ wrote:
  data\SIM_GRID\CLOSURE\adj_mx_diag0.pkl
  data\SIM_GRID\CLOSURE\adj_mx_diag1.pkl
sensors: 120, adj shape: (120, 120)


# Cleansing Hague data

K159-2018-7-1

In [3]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("data/Hauge/K159/K159-2018-7-1.csv", delimiter=';')

# Convert the first column to datetime, assuming the format "day-month-year hours:minutes"
# Note: Adjust the 'dayfirst=True' parameter if your date format varies
first_column_name = df.columns[0]
df[first_column_name] = pd.to_datetime(df[first_column_name], dayfirst=True)

# Format the date in the desired output format "year-month-day hours:minutes"
df[first_column_name] = df[first_column_name].dt.strftime('%Y-%m-%d %H:%M')

ValueError: time data "2018-07-13 00:00" doesn't match format "%Y-%d-%m %H:%M", at position 3451. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
df.to_csv("K159-2018-7-1_modified.csv", sep=';', index=False)

K159-2018-7-2

In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("data/Hauge/K159/K159-2018-7-2.csv", delimiter=';')
df.loc[:len(df)-2, first_column_name] = pd.to_datetime(df[first_column_name].iloc[:-1], dayfirst=True).dt.strftime('%Y-%m-%d %H:%M')

In [ ]:
df.to_csv("K159-2018-7-2_modified.csv", sep=';', index=False)

K182-2019-9-1

In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("data/Hauge/K182/K182-2019-9-1.csv", delimiter=';')

# Convert the first column to datetime, assuming the format "day-month-year hours:minutes"
# Note: Adjust the 'dayfirst=True' parameter if your date format varies
first_column_name = df.columns[0]
df[first_column_name] = pd.to_datetime(df[first_column_name], dayfirst=True)

# Format the date in the desired output format "year-month-day hours:minutes"
df[first_column_name] = df[first_column_name].dt.strftime('%Y-%m-%d %H:%M')

In [ ]:
df.to_csv("K182-2019-9-1_modified.csv", sep=';', index=False)

K182-2019-9-2

In [ ]:
df = pd.read_csv("data/Hauge/K182/K182-2019-9-2.csv", delimiter=';')
df.loc[:len(df)-2, first_column_name] = pd.to_datetime(df[first_column_name].iloc[:-1], dayfirst=True).dt.strftime('%Y-%m-%d %H:%M')

In [ ]:
df.to_csv("K182-2019-9-2_modified.csv", sep=';', index=False)

# Create adjacency matrix HAUGE

In [ ]:
sensors_dict = {
        'K502': ['082', '051', '111', '081'],
        'K504': ['08_2', '12_1', '02_1', '08_1', '11_1', '09_1', '12_2', '02_2', '04_2', '05_1', '04_1', '05_2'],
        'K503': ['02_1', '03_1', '11_1', '07_1', '09_1', '05_1', '05_2', '11_2'],
        'K263': ['052', '041', '061', '112', '051', '031', '111', '081'],
        'K556': ['08_2', '02_1', '08_1', '11_1', '02_2', '10_1', '05_1', '04_1'],
        'K557': ['08_2', '02_1', '08_1', '11_1', '02_2', '10_1'],
        'K559': ['08_2', '12_1', '02_1', '08_1', '02_2', '10_1'],
        'K561': ['08_2', '01_1', '02_1', '08_1', '09_1', '02_2', '10_1', '05_1', '05_2', '06_1'],
        'K198': ['052', '061', '112', '021', '051', '111', '081'],
        'K704': ['12_1', '02_1', '69_1', '11_1', '65_2', '11_3', '65_1', '11_2'],
        'K702': ['03_1', '11_1', '05_1', '04_1', '05_2', '11_2'],
        'K703': ['08_2', '02_1', '08_1', '03_1', '09_1', '05_1', '05_2'],
        # 'K159': ['021', '051', '713', '111', '081'],
        'K159': ['021', '051', '111', '081'],
        'K182': ['621', '682', '681', '622', '121'],
        'K183': ['082', '021', '051', '022', '111', '081'],
        'K128': ['131', '161', '061', '112', '162', '062'],
        # 'K139': ['101', '082', '021', '022', '121', '081'],
        # 'K104': ['082', '021', '051', '022', '111', '081'],
        'K101': ['621', '082', '051', '711', '622', '121', '081'],
        'K206': ['091', '052', '101', '082', '061', '021', '051', '022', '092', '081'],
        'K074': ['052', '051'],
        'K414': ['091', '122', '082', '051', '011', '012', '121', '092', '081'],
        'K415': ['061', '021', '081', '041'],
        'K250': ['091', '101', '102', '021', '022', '092', '081']
        }

# Base path for the sensor data
base_path = 'data/Hauge'

Identify missing timestamps

In [ ]:
# Generate a complete DateTimeIndex from 2018-01-01 00:00:00 to 2019-12-31 23:55:00 with 5-minute intervals
complete_index = pd.date_range(start='2018-01-01 00:00:00', end='2019-12-31 23:55:00', freq='5T')

intersections = []
missing_timestamps_dict = {}  # Initialize a dictionary to store missing timestamps for each sensor

for sensor, codes in sensors_dict.items():
    sensor_folder_path = os.path.join(base_path, sensor)
    csv_files = glob.glob(f"{sensor_folder_path}/{sensor}-201[89]-*-*.csv")
    all_data = []
    for file in tqdm(csv_files, desc=f"Processing {sensor}"):
        # Read the CSV file
        df = pd.read_csv(file, delimiter=';', parse_dates=[0], index_col=0)
        
        # Clean the column names of quotes
        df.columns = [col.replace('"', '').replace("'", "") for col in df.columns]
        df = df[~df.index.astype(str).str.contains('totaal')]
    
        df.index.name = None
        df.index = pd.to_datetime(df.index)
        
        # Keep only the filtered columns
        df = df.loc[:, codes]
        
        # Append the DataFrame to the list
        all_data.append(df)

    # Concatenate all DataFrames into a single DataFrame
    final_df = pd.concat(all_data)

    # Sort the DataFrame by the index (date and time)
    final_df.sort_index(inplace=True)
    final_df.columns = [sensor + "-" + col for col in final_df.columns]
    final_df = final_df[final_df.index.year.isin([2018, 2019])]
    
    # Identify missing timestamps
    missing_timestamps = complete_index.difference(final_df.index)
    
    # Store the missing timestamps in the dictionary
    missing_timestamps_dict[sensor] = missing_timestamps

    # Print the first few rows to verify
    # print(final_df.head())
#     intersections.append(final_df)

# hague = pd.concat(intersections, axis=1)
# hague = hague.sort_index(axis=1)
# missing_timestamps_dict

save the missing timestamp

In [ ]:
import json

# Convert datetime objects to strings in missing_timestamps_dict
for sensor, timestamps in missing_timestamps_dict.items():
    missing_timestamps_dict[sensor] = [timestamp.isoformat() for timestamp in timestamps]

# Save missing_timestamps_dict to a JSON file
with open('missing_timestamps.json', 'w') as f:
    json.dump(missing_timestamps_dict, f, indent=4)

# The file missing_timestamps.json now contains the missing timestamps data

Prepare and generate the standardized data file

In [ ]:
intersections = []

for sensor, codes in sensors_dict.items():
    sensor_folder_path = os.path.join(base_path, sensor)
    print(sensor_folder_path)

    # List all CSV files in the sensor folder for the date range 2018-01 to 2019-12
    csv_files = glob.glob(f"{sensor_folder_path}/{sensor}-201[89]-*-*.csv")
    all_data = []
    for file in csv_files:
        # Read the CSV file
        df = pd.read_csv(file, delimiter=';', parse_dates=[0], index_col=0)
        
        # Clean the column names of quotes
        df.columns = [col.replace('"', '').replace("'", "") for col in df.columns]
        df = df[~df.index.astype(str).str.contains('totaal')]
    
        df.index.name = None
        df.index = pd.to_datetime(df.index)
        # print(df.head())
        
        # Keep only the filtered columns
        df = df.loc[:, codes]
        # print(df.head())
        
        # Append the DataFrame to the list
        all_data.append(df)

    # Concatenate all DataFrames into a single DataFrame
    final_df = pd.concat(all_data)

    # Sort the DataFrame by the index (date and time)
    final_df.sort_index(inplace=True)
    final_df.columns = [sensor + "-" + col for col in final_df.columns]
    final_df = final_df[final_df.index.year.isin([2018, 2019])]
    

    # Print the first few rows to verify
    # print(final_df.head())
    non_unique_indices = final_df.index.duplicated(keep=False)
    non_unique_rows = final_df[non_unique_indices]

    if not non_unique_rows.empty:
        print(non_unique_rows)
    intersections.append(final_df)
# hague = pd.concat(intersections, axis=1)
# hague = hague.sort_index(axis=1)

# hague.head()

In [ ]:
hague = pd.concat(intersections, axis=1)
hague = hague.sort_index(axis=1)

hague.head()

In [ ]:
hague.to_csv("data/Hauge/hague.csv")

### Try to impute the missing values

Only select the columns with missing values < 34%

In [ ]:
nan_percentage = hague.isna().mean() * 100

In [ ]:
print(nan_percentage.sort_values(ascending=False)[:25])

In [ ]:
columns_below_34_nan = nan_percentage[nan_percentage < 34].index.tolist()

# Selecting the columns in 'hague' DataFrame where the NaN percentage is less than 34%
filtered_hague = hague[columns_below_34_nan]

Imputation based on https://www.geo.fu-berlin.de/en/v/soga-py/Advanced-statistics/time-series-analysis/Dealing-with-missing-values/Imputing-missing-values/index.html

In [ ]:
temp_NA_inter = filtered_hague.interpolate(method="time").copy()

In [ ]:
temp_NA_bfill = temp_NA_inter.fillna(method="bfill")

In [ ]:
temp_NA_bfill.to_hdf('data/Hauge/hague_filled.h5', key='df', mode='w')

# Create new adjacency matrix Hague

In [ ]:
distances_df = pd.read_csv('data/Hauge/distances.csv')
df = pd.read_hdf('data/Hauge/hague_filled.h5')
df.head()

In [ ]:
sensor_ids = df.columns.astype(str).tolist()

# Efficiently convert sensor IDs to index for quick lookup (now all as strings)
sensor_id_to_index = {str(sensor_id): index for index, sensor_id in enumerate(sensor_ids)}

# Initialize an empty distance matrix
num_sensors = len(sensor_ids)
dist_mx_array = np.full((num_sensors, num_sensors), np.inf)

# Make sure 'from' and 'to' in distances_df are also strings
distances_df['from'] = distances_df['from'].astype(str)
distances_df['to'] = distances_df['to'].astype(str)

In [ ]:
# Populate the distance matrix with the data from distances_df
for _, row in distances_df.iterrows():
    from_sensor = row['from']
    to_sensor = row['to']
    if from_sensor in sensor_id_to_index and to_sensor in sensor_id_to_index:
        i = sensor_id_to_index[from_sensor]
        j = sensor_id_to_index[to_sensor]
        dist_mx_array[i, j] = row['cost']

# Replace the diagonal with zeros since the distance from a sensor to itself is zero
np.fill_diagonal(dist_mx_array, 0)


In [ ]:
def get_adjacency_matrix(dist_mx_array, normalized_k=0.1):
    # Calculate the standard deviation as theta for normalization
    distances = dist_mx_array[~np.isinf(dist_mx_array)].flatten()
    std = distances.std()
    adj_mx_array = np.exp(-np.square(dist_mx_array / std))
    
    # Set entries below a threshold to zero for sparsity
    adj_mx_array[adj_mx_array < normalized_k] = 0
    return adj_mx_array

In [ ]:
adj_mx_array = get_adjacency_matrix(dist_mx_array).astype(np.float32)

# Construct the adj_mx list as specified with the updated adj_mx_array
adj_mx = [sensor_ids, sensor_id_to_index, adj_mx_array]

In [ ]:
# Construct the file path
directory = 'data/Hauge'
file_name = 'adj_mx.pkl'
file_path = os.path.join(directory, file_name)

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# Save the adj_mx to a .pkl file
with open(file_path, 'wb') as f:
    pickle.dump(adj_mx, f)

# Create a smaller size of the Hague data

In [ ]:
import pandas as pd
df = pd.read_hdf('data/Hauge/hague_filled.h5')

In [ ]:
total_rows = len(df)

# Calculate indices for 12.5th and 87.5th percentiles
start_index = int(total_rows * 0.125)
end_index = int(total_rows * 0.875)

# Select the middle 75% of the rows
middle_75_df = df.iloc[start_index:end_index]
middle_75_df.head()

In [ ]:
middle_75_df.to_hdf('data/Hauge/hague_filled_75.h5', key='df', mode='w')

# Transform Hauge to a compressed sensor datasets

In [ ]:
file_path = 'data/Hauge/Hague_comp_sensor_ref.xlsx'
df1 = pd.read_excel(file_path)
df2 = pd.read_csv('data/Hauge/hague.csv', index_col=0)

In [ ]:
def process_dataframes(df1, df2):
    # New dataframe to hold results
    df3 = pd.DataFrame()
    
    # Iterate over rows in df1
    for idx, row in df1.iterrows():
        intersection = row['Intersection']
        original_columns = str(row['Original']).split(',')

        # Create full column names in df2 and sum them
        if len(original_columns) > 1:
            new_column_name = intersection + '-' + row['New']
            # Create a list of full names to search in df2
            df2_column_names = [intersection + '-' + orig for orig in original_columns]
            # Average the columns in df2 that are in our list if they exist
            df3[new_column_name] = df2[df2_column_names].mean(axis=1)
        else:
            # Just copy the column from df2
            df2_column_name = intersection + '-' + original_columns[0]
            df3[df2_column_name] = df2[df2_column_name] if df2_column_name in df2.columns else pd.Series()
    
    return df3

In [ ]:
df3 = process_dataframes(df1, df2)
df3.to_csv("data/Hauge/hague_comp.csv")

In [ ]:
nan_percentage = df3.isna().mean() * 100
print(nan_percentage.sort_values(ascending=False)[:25])

In [ ]:
columns_below_34_nan = nan_percentage[nan_percentage < 30].index.tolist()

# Selecting the columns in 'hague' DataFrame where the NaN percentage is less than 34%
filtered_hague = df3[columns_below_34_nan]
filtered_hague.index = pd.to_datetime(filtered_hague.index)

In [ ]:
temp_NA_inter = filtered_hague.interpolate(method="time").copy()
temp_NA_bfill = temp_NA_inter.fillna(method="bfill")
temp_NA_bfill.to_hdf('data/Hauge/hague_comp_filled.h5', key='df', mode='w')

# New adj based on combined sensors

In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv('data/HAUGE/distances.csv')
df2 = pd.read_hdf('data/Hauge/hague_comp_filled.h5')
df3 = pd.read_excel('data/Hauge/Hague_comp_sensor_ref.xlsx')
df4 = pd.read_csv('data/Hauge/location_std.csv')

In [ ]:
def transform_id(sensor_id, trajectory, df3):
    # Split the sensor_id by '-'
    parts = sensor_id.split('-')
    if len(parts) != 2:
        return None  # Return None if sensor_id does not match expected format
    
    # Find matching row in df3 with the same trajectory
    mask = (df3['Intersection'] == parts[0]) & (df3['Representation'] == parts[1]) & (df3['Trajectory'] == trajectory)
    if mask.any():
        # If a matching row is found, replace with the 'New' value
        return parts[0] + '-' + df3.loc[mask, 'New'].values[0]
    return None

In [ ]:
def apply_transform_or_keep_original(row, df3):
    trajectories = row['trajectory'].split('+')
    new_ids = []

    for trajectory in trajectories:
        new_id = transform_id(row['sensor_id'], trajectory, df3)
        if new_id is not None:
            new_ids.append(new_id)

    # If no new ID was found for either trajectory, keep the original sensor_id
    if not new_ids:
        new_ids.append(row['sensor_id'])

    # Return a list of dictionaries for each new_id to expand into new rows
    return [{'sensor_id': new_id, 'latitude': row['latitude'], 'longitude': row['longitude'], 'trajectory': trajectory} for new_id, trajectory in zip(new_ids, trajectories)]

# Apply the transformation to generate new rows
expanded_rows = df4.apply(lambda row: apply_transform_or_keep_original(row, df3), axis=1)

# Flatten the list of lists into a single DataFrame
df4_expanded = pd.DataFrame([item for sublist in expanded_rows for item in sublist])

# Keep only the sensor IDs that are present in the columns of df2
df2_columns = set(df2.columns)
df4_filtered = df4_expanded[df4_expanded['sensor_id'].isin(df2_columns)]
df4_filtered

In [ ]:
df4_filtered.to_csv('data/Hauge/location_std_comp.csv', index=False)

In [ ]:
df1['new_from'] = df1.apply(lambda row: transform_id(row['from'], row['trajectory'], df3), axis=1)
df1['new_to'] = df1.apply(lambda row: transform_id(row['to'], row['trajectory'], df3), axis=1)

In [ ]:
df2_columns = set(df2.columns)
df4 = df1[(df1['new_from'].isin(df2_columns)) & (df1['new_to'].isin(df2_columns))]
df4.head()

In [ ]:
df4['from'] = df4['new_from']
df4['to'] = df4['new_to']
df4 = df4.drop(['new_from', 'new_to'], axis=1)
df4.head()

In [ ]:
df4.to_csv('data/Hauge/distances_comp.csv', index=False)

## Generate the adjacency matrix for the combined sensors

In [ ]:
distances_df = pd.read_csv('data/Hauge/distances_comp.csv')
df = pd.read_hdf('data/Hauge/hague_comp_filled.h5')

sensor_ids = df.columns.astype(str).tolist()

# Efficiently convert sensor IDs to index for quick lookup (now all as strings)
sensor_id_to_index = {str(sensor_id): index for index, sensor_id in enumerate(sensor_ids)}

# Initialize an empty distance matrix
num_sensors = len(sensor_ids)
dist_mx_array = np.full((num_sensors, num_sensors), np.inf)

# Make sure 'from' and 'to' in distances_df are also strings
distances_df['from'] = distances_df['from'].astype(str)
distances_df['to'] = distances_df['to'].astype(str)

# Populate the distance matrix with the data from distances_df
for _, row in distances_df.iterrows():
    from_sensor = row['from']
    to_sensor = row['to']
    if from_sensor in sensor_id_to_index and to_sensor in sensor_id_to_index:
        i = sensor_id_to_index[from_sensor]
        j = sensor_id_to_index[to_sensor]
        dist_mx_array[i, j] = row['cost']

# Replace the diagonal with zeros since the distance from a sensor to itself is zero
np.fill_diagonal(dist_mx_array, 0)

In [ ]:
def get_adjacency_matrix(dist_mx_array, normalized_k=0.1):
    # Calculate the standard deviation as theta for normalization
    distances = dist_mx_array[~np.isinf(dist_mx_array)].flatten()
    std = distances.std()
    adj_mx_array = np.exp(-np.square(dist_mx_array / std))
    
    # Set entries below a threshold to zero for sparsity
    adj_mx_array[adj_mx_array < normalized_k] = 0
    return adj_mx_array

In [ ]:
adj_mx_array = get_adjacency_matrix(dist_mx_array).astype(np.float32)

# Construct the adj_mx list as specified with the updated adj_mx_array
adj_mx = [sensor_ids, sensor_id_to_index, adj_mx_array]
adj_mx[2][0]

In [ ]:
# Construct the file path
directory = 'data/Hauge'
file_name = 'adj_mx_comp1.pkl'
file_path = os.path.join(directory, file_name)

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# Save the adj_mx to a .pkl file
with open(file_path, 'wb') as f:
    pickle.dump(adj_mx, f)

## Generate the adjacency matrix for the combined sensors with direction applied to only count in foward direction that are connected by lanes

In [ ]:
distances_df = pd.read_csv('data/Hauge/distances_comp_directed.csv')
df = pd.read_hdf('data/Hauge/hague_comp_filled.h5')

sensor_ids = df.columns.astype(str).tolist()

# Efficiently convert sensor IDs to index for quick lookup (now all as strings)
sensor_id_to_index = {str(sensor_id): index for index, sensor_id in enumerate(sensor_ids)}

# Initialize an empty distance matrix
num_sensors = len(sensor_ids)
dist_mx_array = np.full((num_sensors, num_sensors), np.inf)

# Make sure 'from' and 'to' in distances_df are also strings
distances_df['from'] = distances_df['from'].astype(str)
distances_df['to'] = distances_df['to'].astype(str)

# Populate the distance matrix with the data from distances_df
for _, row in distances_df.iterrows():
    from_sensor = row['from']
    to_sensor = row['to']
    if from_sensor in sensor_id_to_index and to_sensor in sensor_id_to_index:
        i = sensor_id_to_index[from_sensor]
        j = sensor_id_to_index[to_sensor]
        dist_mx_array[i, j] = row['cost']

# Replace the diagonal with zeros since the distance from a sensor to itself is zero
np.fill_diagonal(dist_mx_array, 0)

In [ ]:
def get_adjacency_matrix(dist_mx_array, normalized_k=0.1):
    # Calculate the standard deviation as theta for normalization
    distances = dist_mx_array[~np.isinf(dist_mx_array)].flatten()
    std = distances.std()
    adj_mx_array = np.exp(-np.square(dist_mx_array / std))
    
    # Set entries below a threshold to zero for sparsity
    adj_mx_array[adj_mx_array < normalized_k] = 0
    return adj_mx_array

In [ ]:
adj_mx_array = get_adjacency_matrix(dist_mx_array).astype(np.float32)

# Construct the adj_mx list as specified with the updated adj_mx_array
adj_mx = [sensor_ids, sensor_id_to_index, adj_mx_array]
adj_mx[2][0]

In [ ]:
# Construct the file path
directory = 'data/Hauge'
file_name = 'adj_mx_comp2.pkl'
file_path = os.path.join(directory, file_name)

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# Save the adj_mx to a .pkl file
with open(file_path, 'wb') as f:
    pickle.dump(adj_mx, f)

# Save a copy of adjacency matrix to csv file for inspection

In [ ]:
import numpy as np
import pickle

METR-LA

In [ ]:
file_path = 'data/METRLA/adj_mx.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/METRLA/adj_mx.csv", adj_mx[2], delimiter=",", fmt='%.4f')

In [ ]:
file_path = 'data/METRLA/adj_mx_new1.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/METRLA/adj_mx_new1.csv", adj_mx[2], delimiter=",", fmt='%.4f')

PeMS-BAY

In [ ]:
file_path = 'data/PEMSBAY/adj_mx_bay.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/PEMSBAY/adj_mx_bay.csv", adj_mx[2], delimiter=",", fmt='%.4f')

In [ ]:
file_path = 'data/PEMSBAY/adj_mx_new1.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/PEMSBAY/adj_mx_new1.csv", adj_mx[2], delimiter=",", fmt='%.4f')

Hauge

In [ ]:
file_path = 'data/Hauge/adj_mx1.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/Hauge/adj_mx1.csv", adj_mx[2], delimiter=",", fmt='%.4f')

In [ ]:
file_path = 'data/Hauge/adj_mx_comp1.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/Hauge/adj_mx_comp1.csv", adj_mx[2], delimiter=",", fmt='%.4f')

In [ ]:
file_path = 'data/Hauge/adj_mx_comp2.pkl'
with open(file_path, 'rb') as f:
    adj_mx = pickle.load(f, encoding='latin1')
np.savetxt("data/Hauge/adj_mx_comp2.csv", adj_mx[2], delimiter=",", fmt='%.4f')

# A even smaller size dataset Hague for cost model (memory saver)

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_hdf('data/Hauge/hague_comp_filled.h5')

Middle 20

In [ ]:
# Calculate indices for the middle 20%
start_idx = int(len(df) * 0.40)
end_idx = int(len(df) * 0.60)

# Select the middle 20% chunk
df_middle_20_percent = df.iloc[start_idx:end_idx]

# Convert index to datetime
df_middle_20_percent.index = pd.to_datetime(df_middle_20_percent.index, unit='ns')

# Save to HDF5 file
df_middle_20_percent.to_hdf('data/Hauge/hague_comp_filled_20_2.h5', key='df', mode='w')

First 20

In [ ]:
# Calculate indices for the middle 20%
start_idx = int(len(df) * 0.0)
end_idx = int(len(df) * 0.2)

# Select the middle 20% chunk
df_first_20_percent = df.iloc[start_idx:end_idx]

# Convert index to datetime
df_first_20_percent.index = pd.to_datetime(df_first_20_percent.index, unit='ns')

# Save to HDF5 file
df_first_20_percent.to_hdf('data/Hauge/hague_comp_filled_20_3.h5', key='df', mode='w')

# Locations of all sensors in 3 datasets

In [ ]:
import pandas as pd
import folium

# Load the CSV data
csv_file_path = 'data/METRLA/graph_sensor_locations.csv'
data = pd.read_csv(csv_file_path)

# Create a folium map centered around the mean latitude and longitude
center_lat = data['latitude'].mean()
center_lon = data['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='OpenStreetMap')

# Add sensor locations to the map
for idx, row in data.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7,
        popup=f'Sensor ID: {row["sensor_id"]}'
    ).add_to(m)

# Save the map to an HTML file
m.save('sensor_locations_map_METRLA.html')

m

In [ ]:
import pandas as pd
import folium

# Load the CSV data
csv_file_path = 'data/PEMSBAY/graph_sensor_locations_bay.csv'
data = pd.read_csv(csv_file_path)

# Create a folium map centered around the mean latitude and longitude
center_lat = data['latitude'].mean()
center_lon = data['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='OpenStreetMap')

# Add sensor locations to the map
for idx, row in data.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7,
        popup=f'Sensor ID: {row["sensor_id"]}'
    ).add_to(m)

# Save the map to an HTML file
m.save('sensor_locations_map_bay.html')

m

In [ ]:
import pandas as pd
import folium

# Load the CSV data
csv_file_path = 'data/Hauge/location_std.csv'
data = pd.read_csv(csv_file_path)

# Extract the first trajectory if multiple are concatenated
data['trajectory'] = data['trajectory'].apply(lambda x: x.split('+')[0])

# Predefined colors for the trajectories
color_mapping = {
    'T1N': 'red',
    'T1S': 'orange',
    'T2N': 'blue',
    'T2S': 'purple'
}

# Create a folium map centered around the mean latitude and longitude
center_lat = data['latitude'].mean()
center_lon = data['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='OpenStreetMap')

# Add sensor locations to the map with specific colors for each trajectory
for idx, row in data.iterrows():
    trajectory = row['trajectory']
    color = color_mapping.get(trajectory, 'gray')  # Default to gray if trajectory is not in color_mapping
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f'Sensor ID: {row["sensor_id"]}, Trajectory: {row["trajectory"]}'
    ).add_to(m)

# Save the map to an HTML file
m.save('sensor_locations_map_Hauge.html')

m